In [1]:
%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection'

In [4]:
import tensorflow as tf

In [10]:
def build_model(input_shape, classes, learning_rate):
    """
    Builds a State-of-the-Art Transfer Learning model using EfficientNetB3.
    Includes regularization layers designed to prevent overfitting on clinical image data.
    """
    # Explicitly use Input layer to resolve the Keras KerasTensor / Sequential warnings
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    # 1. Base Model: Pre-trained on ImageNet to capture rich edge/texture features
    # Note: Keras EfficientNet has built-in Rescaling, no separate normalization layer needed.
    base_model = tf.keras.applications.EfficientNetB3(
        weights='imagenet', 
        include_top=False, 
        input_tensor=inputs
    )
    
    # Freeze the base model to preserve features during the initial phase
    base_model.trainable = False
    
    # 2. Custom Classification Head
    x = base_model.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    
    # Dense layer for feature consolidation with L2 regularization
    x = tf.keras.layers.Dense(
        256, 
        activation='relu', 
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    )(x)
    
    # Dropout to heavily combat overfitting on smaller medical image datasets
    x = tf.keras.layers.Dropout(0.4)(x)
    
    # Output layer
    outputs = tf.keras.layers.Dense(classes, activation='softmax')(x)
    
    model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
    
    # Compile with Adam optimizer using a slightly lower LR for stability in transfer learning
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_classes: int

In [12]:
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:
        config = self.config.training
        params = self.params

        create_directories([config.root_dir])

        training_config = TrainingConfig(
            root_dir=Path(config.root_dir),
            # Migrating to the modern .keras extension to remove the legacy HDF5 warning
            trained_model_path= Path(config.trained_model_path),
            training_data= Path(config.training_data),
            params_epochs= params.EPOCHS,
            params_batch_size= params.BATCH_SIZE,
            params_is_augmentation= params.AUGMENTATION,
            params_image_size= params.IMAGE_SIZE,
            params_classes= params.CLASSES    
        )

        return training_config

In [13]:
# 3. Model Training Manager
class ModelTrainer:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def train(self):
        # 1. Load the transformed datasets
        train_ds = tf.data.Dataset.load(str(self.config.training_data))
        
        # 2. Build the state-of-the-art model
        # Recommended image dimensions for EfficientNetB3 are (300, 300, 3) 
        # Make sure to set IMAGE_SIZE: [300, 300, 3] in your params.yaml if possible
        model = build_model(
            input_shape=tuple(self.config.params_image_size),
            classes=self.config.params_classes,
            learning_rate=0.0001  # Lowered slightly for stable transfer learning
        )
        
        # 3. Fit the model
        model.fit(
            train_ds,
            epochs=self.config.params_epochs
        )
        
        # 4. Save model using native Keras v3 format
        model.save(self.config.trained_model_path)
        logger.info(f"Model successfully saved to {self.config.trained_model_path}")

In [14]:
# 4. Pipeline Execution
if __name__ == "__main__":
    STAGE_NAME = "Model Training Stage"
    try:
        logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
        config = ConfigurationManager()
        training_config = config.get_training_config()
        model_trainer = ModelTrainer(config=training_config)
        model_trainer.train()
        logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
    except Exception as e:
        logger.exception(e)
        raise e

[2026-07-11 22:58:09,013: INFO: 23487130: >>>>>> stage Model Training Stage started <<<<<<]
[2026-07-11 22:58:09,020: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 22:58:09,023: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-11 22:58:09,025: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-07-11 22:58:09,026: INFO: common: created directory at: artifacts]
[2026-07-11 22:58:09,028: INFO: common: created directory at: artifacts/training]
Epoch 1/25
 11/435 ━━━━━━━━━━━━━━━━━━━━ 9:33 1s/step - accuracy: 0.0483 - loss: 4.0269

KeyboardInterrupt: 